# CGNS — reading a NASA aircraft CFD grid

**CGNS** is a standard file format for CFD (computational fluid dynamics) meshes. This notebook reads a NASA
**Common Research Model (CRM)** grid — an aircraft sitting inside a simulation domain — with the `cgns` driver,
and shows what is in it.

It is a **3-D** grid, which matters for how we look at it (more on that below).

## 1. Read the file

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from cleopatra.config import Config

from pyramids.netcdf.ugrid import MeshDataset

DATA = Path("../../../examples/data/real-data")
grid = DATA / "cgns" / "cgns_dpw4crm_unstructured_mixed_g0-coarse_2.07mn_orphan98.cgns"

ds = MeshDataset.read_file(grid, driver="cgns")
Config.set_matplotlib_backend()   # let cleopatra pick the inline backend
print("nodes:", ds.n_node)
print("faces:", ds.n_face)

The 263 MB file reads into a pyramids `MeshDataset` in about a second — **50 316 nodes** and **58 480 faces**.

The grid has two parts:

- the **aircraft surface** (the thing we care about), and
- the large **simulation box** around it — the outer boundary of the airflow domain.

Because it is a 3-D surface, the height of each point is kept as a node field called **`node_z`**.

## 2. The grid from above

pyramids works in 2-D, so we look at the grid's points from straight above (the x–y plane) and colour each
point by its height `z`. (The mesh is 3-D, so instead of a flat colour-map we just plot the points.)

In [ ]:
x = np.asarray(ds.mesh.node_x)
y = np.asarray(ds.mesh.node_y)
z = np.asarray(ds.masked_values("node_z"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x, y, c=z, cmap="viridis", s=1, linewidths=0)
ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("CGNS grid from above")

The sparse regular dots across the whole frame are the **simulation box**. The **aircraft** is the small,
dense cluster near the bottom-centre — it is a tiny part of the whole domain. That is normal for a CFD grid:
most of the mesh is the space *around* the object, not the object itself.

## 3. Zoom in on the aircraft

`crop(...)` keeps only the faces inside a box, so we can cut down to the aircraft and look again.

In [ ]:
aircraft = ds.crop(bbox=(-0.2, -0.05, 1.5, 0.6))

ax_x = np.asarray(aircraft.mesh.node_x)
ax_y = np.asarray(aircraft.mesh.node_y)
ax_z = np.asarray(aircraft.masked_values("node_z"))

fig, ax = plt.subplots(figsize=(8, 3))
sc = ax.scatter(ax_x, ax_y, c=ax_z, cmap="coolwarm", s=6, linewidths=0)
plt.colorbar(sc, label="height z")
ax.set_aspect("equal")
ax.set_xlim(-0.2, 1.5); ax.set_ylim(-0.05, 0.5)   # frame the aircraft, drop stray box points
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Zoomed to the aircraft")

Now the shape is recognisable: the **fuselage** runs along `y = 0` and the **wing** spreads out from it
(only half the aircraft is modelled, because it is left–right symmetric). The colours are the surface height.

## Takeaway

- The `cgns` driver reads a large CFD container (**263 MB here**) into an ordinary pyramids `MeshDataset` in
  about a second.
- The grid is **3-D** — an aircraft surface inside a simulation box — so we view it as **points from above**
  rather than a flat colour-map (a flat map of a 3-D shape is not meaningful).
- `crop(...)` lets us zoom from the whole domain down to just the aircraft, and every pyramids operation works
  the same on the cropped mesh as on the full one.